In [ ]:
from abc import ABC, abstractmethod
import pyspark.sql.functions as F

In [ ]:
class FactTaxiTransformer(ABC):

    def __init__(self, fact_table):
        self.sources = ClassManagement.get_all_subclass_instances(TaxiSilverTable)
        self.fact_table = fact_table

    @abstractmethod
    def enrich_data(self, data):
        pass
    
    def get_new_data(self, source):
        new_months = self.fact_table.new_months(source, source.TAXI_TYPE_ID).df
        data = source.enrich_by_year_month()\
            .filter_by_year_months(new_months)\
            .drop_year_month()\
            .enrich_by_value_columns()\
            .enrich_by_taxi_type_id()\
            .enrich_by_date()
        return self.enrich_data(data).df

    def aggregate(self, df):
        return df.groupBy(*self.fact_table.PKS)\
            .agg(F.count("*").alias("trip_count"),
                F.sum("trip_distance").alias("total_distance"),
                F.avg("trip_distance").alias("avg_trip_distance"),
                F.sum("fare_amount").alias("total_fare"),
                F.avg("fare_amount").alias("avg_fare"),
                F.sum("total_amount").alias("total_amount"),
                F.avg("total_amount").alias("avg_total_amount"),
                F.sum("tip_amount").alias("total_tip"),
                F.avg("tip_amount").alias("avg_tip"))

    def cast_columns(self, df):
        new_df = df
        for column in ["pickup_location_id", "dropoff_location_id"]:
            new_df = new_df.withColumn(column, F.col(column).cast("int"))
        return new_df

    def transform(self):
        for source in self.sources:
            new_data = self.get_new_data(source)
            aggregated_data = self.aggregate(new_data)
            aggregated_data = self.cast_columns(aggregated_data)
            if not aggregated_data.isEmpty():
                self.fact_table.__class__(aggregated_data).append()